In [1]:
import os
import pandas as pd
from functools import partial

In [2]:
application = pd.read_csv('../data/dseb63_application_train.csv', index_col=0)
application2 = pd.read_csv('../data/dseb63_application_test.csv', index_col=0) 
previous_application = pd.read_csv('../data/dseb63_previous_application.csv')

In [3]:
application.head()

,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,SK_ID_CURR
0,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,278621
1,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,139008
2,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,138348
3,0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,454500.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0,64140
4,0,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,1530000.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,219374


In [4]:
previous_application.head()

,SK_ID_PREV,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,...,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL,SK_ID_CURR
0,2030495,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,...,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0,293189
1,1696966,Consumer loans,68258.655,1800000.0,1754721.0,180000.0,1800000.0,SATURDAY,18,Y,...,36.0,low_normal,POS industry with interest,NaN,NaN,NaN,NaN,NaN,NaN,293189
2,2154916,Consumer loans,12417.390,108400.5,119848.5,0.0,108400.5,SUNDAY,14,Y,...,12.0,middle,POS industry with interest,365243.0,-512.0,-182.0,-392.0,-387.0,0.0,293189
3,2802425,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,...,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0,91587
4,1536272,Cash loans,21709.125,450000.0,512370.0,NaN,450000.0,WEDNESDAY,9,Y,...,36.0,low_normal,Cash X-Sell: low,365243.0,-485.0,565.0,-155.0,-147.0,1.0,91587


In [5]:
previous_application['SK_ID_CURR'].nunique()

291057

In [6]:


common_columns = [col for col in application.columns if col in previous_application.columns]


application_common1 = application[common_columns]
application_common2 = application2[common_columns]
application_common = pd.concat([application_common1, application_common2])
features = pd.DataFrame({'SK_ID_CURR': application_common['SK_ID_CURR']})


merged_tables = previous_application[common_columns + ['DAYS_DECISION']].merge(application_common, on='SK_ID_CURR',
                                                                               how='right')
merged_tables.head()

,NAME_CONTRACT_TYPE_x,AMT_CREDIT_x,AMT_ANNUITY_x,AMT_GOODS_PRICE_x,NAME_TYPE_SUITE_x,WEEKDAY_APPR_PROCESS_START_x,HOUR_APPR_PROCESS_START_x,SK_ID_CURR,DAYS_DECISION,NAME_CONTRACT_TYPE_y,AMT_CREDIT_y,AMT_ANNUITY_y,AMT_GOODS_PRICE_y,NAME_TYPE_SUITE_y,WEEKDAY_APPR_PROCESS_START_y,HOUR_APPR_PROCESS_START_y
0,Cash loans,1035882.0,98356.995,900000.0,Unaccompanied,FRIDAY,12.0,278621,-746.0,Cash loans,1293502.5,35698.5,1129500.0,Family,MONDAY,11
1,Consumer loans,348637.5,64567.665,337500.0,Family,SUNDAY,17.0,278621,-828.0,Cash loans,1293502.5,35698.5,1129500.0,Family,MONDAY,11
2,Consumer loans,68053.5,6737.310,68809.5,Family,SATURDAY,15.0,278621,-2341.0,Cash loans,1293502.5,35698.5,1129500.0,Family,MONDAY,11
3,Cash loans,675000.0,24246.000,675000.0,Unaccompanied,THURSDAY,15.0,139008,-181.0,Cash loans,312682.5,29686.5,297000.0,Unaccompanied,WEDNESDAY,17
4,Revolving loans,0.0,NaN,NaN,NaN,THURSDAY,15.0,139008,-181.0,Cash loans,312682.5,29686.5,297000.0,Unaccompanied,WEDNESDAY,17


In [7]:
merged_tables['SK_ID_CURR'].nunique()

307511

In [8]:
merged_sorted = merged_tables.sort_values(['SK_ID_CURR', 'DAYS_DECISION'])

In [9]:
merged_sorted['annuity_diff'] = merged_sorted['AMT_ANNUITY_y'] - merged_sorted['AMT_ANNUITY_x']
merged_sorted['annuity_ratio'] = merged_sorted['AMT_ANNUITY_y'] / merged_sorted['AMT_ANNUITY_x'] * (merged_sorted['AMT_ANNUITY_x'] != 0)
merged_sorted['credit_diff'] = merged_sorted['AMT_CREDIT_y'] - merged_sorted['AMT_CREDIT_x']
merged_sorted['credit_ratio'] = merged_sorted['AMT_CREDIT_y'] / merged_sorted['AMT_CREDIT_x'] * (merged_sorted['AMT_CREDIT_x'] != 0)

In [ ]:
merged_sorted['the_same_contract_type'] = (
    merged_sorted['NAME_CONTRACT_TYPE_x'] == merged_sorted['NAME_CONTRACT_TYPE_y']).astype(int)
merged_sorted['the_same_weekday'] = (merged_sorted['WEEKDAY_APPR_PROCESS_START_x'] == merged_sorted['WEEKDAY_APPR_PROCESS_START_y']).astype(int)
merged_sorted['hour_diff'] = merged_sorted['HOUR_APPR_PROCESS_START_x'] - merged_sorted['HOUR_APPR_PROCESS_START_y']
merged_sorted['the_same_type_suite'] = (merged_sorted['NAME_TYPE_SUITE_x'] == merged_sorted['NAME_TYPE_SUITE_y']
                                       ).astype(int)
merged_sorted['the_same_type_suite'][merged_sorted['NAME_TYPE_SUITE_x'].isnull()] = 1

/tmp/ipykernel_503807/2346401584.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  merged_sorted['the_same_type_suite'][merged_sorted['NAME_TYPE_SUITE_x'].isnull()] = 1
/tmp/ipykernel_503807/2346401584.py:7: SettingWithCopyWarning: 
A valu

In [11]:
def _get_last_k_applications_feature_name(feature_name, number, suffix):
    return 'application_previous_application_{}_last_{}_applications_{}'.format(feature_name, number, suffix)


def get_last_k_credits_features(merged_sorted, numbers_of_applications):
    features = pd.DataFrame({'SK_ID_CURR': merged_sorted['SK_ID_CURR'].unique()})
    feature_list = ['annuity_diff', 'annuity_ratio', 'credit_diff', 'credit_ratio', 'the_same_contract_type',
                        'the_same_type_suite', 'the_same_weekday', 'hour_diff']

    for number in numbers_of_applications:
        table_tail = merged_sorted.groupby('SK_ID_CURR').tail(number)
        tail_groupby = table_tail.groupby('SK_ID_CURR')
        g = tail_groupby[feature_list].agg('mean')

        g = g.rename(axis='columns', mapper=partial(_get_last_k_applications_feature_name, number=number,
                                        suffix='mean')).reset_index()

        features = features.merge(g, how='left', on=['SK_ID_CURR'])
    return features

In [12]:
features.shape

(307511, 1)

In [13]:
g = get_last_k_credits_features(merged_sorted, numbers_of_applications=[1,3,5,10])
features = features.merge(g, on=['SK_ID_CURR'], how='left')

In [14]:
features.to_parquet('../data/dseb63_app_prev_app_features.parquet', index=False)

In [15]:
features

,SK_ID_CURR,application_previous_application_annuity_diff_last_1_applications_mean,application_previous_application_annuity_ratio_last_1_applications_mean,application_previous_application_credit_diff_last_1_applications_mean,application_previous_application_credit_ratio_last_1_applications_mean,application_previous_application_the_same_contract_type_last_1_applications_mean,application_previous_application_the_same_type_suite_last_1_applications_mean,application_previous_application_the_same_weekday_last_1_applications_mean,application_previous_application_hour_diff_last_1_applications_mean,application_previous_application_annuity_diff_last_3_applications_mean,...,application_previous_application_the_same_weekday_last_5_applications_mean,application_previous_application_hour_diff_last_5_applications_mean,application_previous_application_annuity_diff_last_10_applications_mean,application_previous_application_annuity_ratio_last_10_applications_mean,application_previous_application_credit_diff_last_10_applications_mean,application_previous_application_credit_ratio_last_10_applications_mean,application_previous_application_the_same_contract_type_last_10_applications_mean,application_previous_application_the_same_type_suite_last_10_applications_mean,application_previous_application_the_same_weekday_last_10_applications_mean,application_previous_application_hour_diff_last_10_applications_mean
0,278621,-62658.495,0.362948,257620.5,1.248697,1.0,0.0,0.0,1.0,-20855.4900,...,0.0,3.666667,-20855.49000,2.071487,8.093115e+05,7.988668,0.333333,0.666667,0.000000,3.666667
1,139008,-3009.600,0.907952,-593932.5,0.344890,1.0,1.0,0.0,-2.0,-3009.6000,...,0.0,-2.000000,6035.32500,3.008889,2.098700e+04,2.782211,0.555556,0.888889,0.000000,-2.333333
2,138348,5827.860,1.363386,238711.5,1.870294,1.0,1.0,0.0,0.0,6679.4250,...,0.2,1.200000,9586.69500,3.887344,3.463612e+05,10.121560,0.666667,0.666667,0.166667,1.333333
3,64140,NaN,NaN,490495.5,NaN,1.0,1.0,0.0,2.0,5919.7950,...,0.0,-4.000000,11677.80375,2.080781,3.277278e+05,5.189250,0.400000,0.800000,0.000000,-4.000000
4,219374,14611.590,1.532038,1269189.0,5.866317,0.0,0.0,0.0,0.0,14611.5900,...,0.0,0.000000,14611.59000,1.532038,1.269189e+06,5.866317,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,150442,NaN,NaN,521280.0,NaN,1.0,1.0,0.0,0.0,-8459.8875,...,0.4,1.000000,-14927.57500,1.152210,-1.378994e+05,2.261173,1.000000,0.300000,0.300000,0.900000
307507,5217,NaN,NaN,900000.0,NaN,0.0,1.0,0.0,0.0,20594.9025,...,0.2,0.600000,23495.31000,2.326698,7.247595e+05,6.875683,0.000000,0.333333,0.333333,0.166667
307508,260741,NaN,NaN,202500.0,NaN,0.0,1.0,0.0,-5.0,14631.7725,...,0.0,-4.200000,10253.40500,2.826527,3.191805e+04,2.442285,0.300000,0.800000,0.000000,-2.900000
307509,284794,-3851.190,0.887419,-141552.0,0.642773,0.0,1.0,0.0,3.0,9337.2975,...,0.0,1.000000,9337.29750,2.381915,2.834550e+04,2.577085,0.000000,1.000000,0.000000,1.000000
